# GPT-5.2 Synthetic Votes: Extra 100k Pairs

Generates 100,000 additional GPT-5.2 synthetic votes on new image pairs
not previously voted on by humans or GPT.

**Why extra pairs?**
More pairwise comparisons per image reduce TrueSkill sigma (ranking uncertainty).
The 25k human and 25k GPT votes cover 25,569 unique pairs from 19,970 images.
This notebook samples 100,000 new pairs from the remaining possible combinations,
ensuring no pair is repeated.

**Toggle:** set `N_PAIRS` in section 2.
- `N_PAIRS = 100`: quick test (~6 minutes)
- `N_PAIRS = None`: full run (100,000 new pairs)

**Output columns:** `user_id`, `image_name1`, `image_name2`, `vote`,
`location`, `voting_time`, `model_id`, `reason`

**Pipeline:**
1. Install dependencies
2. Imports and configuration
3. Load all valid image names and existing voted pairs
4. Generate 100,000 new unique pairs
5. Azure blob connection
6. Final prompt (Option B)
7. GPT API call and vote parser
8. Run pairs
9. Export and summary

## 1. Install dependencies

In [ ]:
import sys, subprocess
for pkg in ['azure-storage-blob', 'openai', 'Pillow', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   check=True)
print('Done.')

## 2. Imports and configuration

**Edit this cell before running.**

- Set `N_PAIRS = 100` for a quick test to verify everything works
- Set `N_PAIRS = None` for the full run (100,000 new pairs)

In [ ]:
import os, io, re, time, base64, uuid, random
import pandas as pd
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient
from openai import AzureOpenAI

# ============================================================
# TOGGLE — number of new pairs to run
# N_PAIRS = 100     -> quick test (~6 minutes)
# N_PAIRS = None    -> full run (100,000 new pairs)
# ============================================================
N_PAIRS = None

# ============================================================
# TARGET — total extra pairs to generate in full run
# ============================================================
N_TARGET = 100_000

# ============================================================
# PATHS
# ============================================================
FEATURES_PATH        = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv'
HUMAN_VOTES_PATH     = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data\human_votes_05_06_26.csv'
GPT_VOTES_25K_PATH   = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_25k_20_06_26.csv'
RESULT_CSV_PATH      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_100k_2_20_06_26.csv'

# ============================================================
# AZURE OPENAI
# ============================================================
OPENAI_MODEL_ID       = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-5.2')
AZURE_OPENAI_ENDPOINT = os.getenv('DEPLOYMENT_NAME',
                                   'https://YOUR_AZURE_OPENAI_RESOURCE.openai.azure.com/')
AZURE_OPENAI_API_KEY  = os.getenv(
    'AZURE_OPENAI_API_KEY',
    'YOUR_AZURE_OPENAI_API_KEY'
)
client = AzureOpenAI(
    api_version    = os.getenv('OPENAI_API_VERSION', '2024-12-01-preview'),
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
)

# ============================================================
# AZURE BLOB
# ============================================================
AZURE_CONN_STR = os.getenv(
    'AZURE_CONN_STR',
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
BLOB_CONTAINER = 'london'
IMAGE_PREFIX   = 'training/training/Images/'

# ============================================================
# OUTPUT SETTINGS
# ============================================================
GPT_USER_ID      = str(uuid.uuid4())
CHECKPOINT_EVERY = 10 if (N_PAIRS is not None and N_PAIRS <= 100) else 500
RANDOM_SEED      = 123   # different from 25k run (seed=42 or 99) to get new pairs
RETRY_MAX        = 3
RATE_LIMIT_SEC   = 0.5

n_to_run  = N_PAIRS if N_PAIRS is not None else N_TARGET
run_label = f'{N_PAIRS} pairs (test)' if N_PAIRS is not None else f'{N_TARGET:,} new pairs (full run)'
print('Configuration loaded.')
print(f'  Model           : {OPENAI_MODEL_ID}')
print(f'  Run             : {run_label}')
print(f'  GPT user_id     : {GPT_USER_ID}')
print(f'  Output CSV      : {RESULT_CSV_PATH}')
print(f'  Checkpoint every: {CHECKPOINT_EVERY} results')

## 3. Load valid image names and existing voted pairs

Valid image names come from the raw features file (19,995 images).
Existing voted pairs are loaded from both the human votes and the 25k GPT votes
already collected. New pairs are sampled only from combinations not in this set.

In [ ]:
# ── load valid image names from features file ─────────────────────────────────
feat_df     = pd.read_csv(FEATURES_PATH, usecols=['image_name'])
all_images  = feat_df['image_name'].apply(
    lambda x: os.path.basename(str(x).strip())
).dropna().unique().tolist()

print(f'Valid images in features file : {len(all_images):,}')

# ── load existing voted pairs ─────────────────────────────────────────────────
def load_pair_keys(path, col1='image_name1', col2='image_name2'):
    df = pd.read_csv(path, usecols=[col1, col2])
    df[col1] = df[col1].apply(lambda x: os.path.basename(str(x).strip()))
    df[col2] = df[col2].apply(lambda x: os.path.basename(str(x).strip()))
    return set(
        tuple(sorted([r[col1], r[col2]]))
        for _, r in df.iterrows()
    )

print('Loading existing voted pairs...')
human_pairs  = load_pair_keys(HUMAN_VOTES_PATH)
gpt_25k_pairs = load_pair_keys(GPT_VOTES_25K_PATH)
existing_pairs = human_pairs | gpt_25k_pairs

print(f'  Human voted pairs      : {len(human_pairs):,}')
print(f'  GPT 25k voted pairs    : {len(gpt_25k_pairs):,}')
print(f'  Total existing pairs   : {len(existing_pairs):,}')

# ── compute how many new pairs are possible ───────────────────────────────────
n_images       = len(all_images)
max_possible   = n_images * (n_images - 1) // 2
max_new        = max_possible - len(existing_pairs)

print(f'\nImage pool size          : {n_images:,}')
print(f'Max possible unique pairs: {max_possible:,}')
print(f'Existing pairs           : {len(existing_pairs):,}')
print(f'New pairs available      : {max_new:,}')
print(f'Target new pairs         : {n_to_run:,}')

if n_to_run > max_new:
    print(f'WARNING: target ({n_to_run:,}) exceeds available new pairs ({max_new:,})')
    print(f'         reducing target to {max_new:,}')
    n_to_run = max_new

## 4. Generate new unique pairs

Pairs are sampled randomly from all possible image combinations,
excluding any pair already voted on by humans or GPT.
Canonical key = tuple(sorted([img1, img2])) ensures no pair appears twice
regardless of image order.

In [ ]:
print(f'Sampling {n_to_run:,} new unique pairs...')

rng        = np.random.RandomState(RANDOM_SEED)
new_pairs  = set()
images_arr = np.array(all_images)
n_imgs     = len(images_arr)

# sample random pairs until we have enough
# use batched sampling for efficiency
batch_size   = min(n_to_run * 5, 500_000)
attempts     = 0
max_attempts = 20

while len(new_pairs) < n_to_run and attempts < max_attempts:
    idx1 = rng.randint(0, n_imgs, size=batch_size)
    idx2 = rng.randint(0, n_imgs, size=batch_size)
    for i, j in zip(idx1, idx2):
        if i == j:
            continue
        key = tuple(sorted([images_arr[i], images_arr[j]]))
        if key not in existing_pairs and key not in new_pairs:
            new_pairs.add(key)
            if len(new_pairs) >= n_to_run:
                break
    attempts += 1
    if len(new_pairs) < n_to_run:
        print(f'  Batch {attempts}: {len(new_pairs):,} / {n_to_run:,} pairs found...')

print(f'  New pairs generated : {len(new_pairs):,}')

# convert to dataframe
sample_df = pd.DataFrame(
    [(a, b) for a, b in new_pairs],
    columns=['image_name1', 'image_name2']
).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

total_pairs = len(sample_df)
print(f'  Sample ready        : {total_pairs:,} pairs')
print(sample_df.head(5).to_string(index=False))

## 5. Azure blob connection

In [ ]:
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

def download_blob_to_pil(blob_name, timeout=30):
    blob_data = container_client.download_blob(
        blob_name, timeout=timeout
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')

print('Testing Azure connection...')
test_blob = IMAGE_PREFIX + sample_df['image_name1'].iloc[0]
try:
    img_test = download_blob_to_pil(test_blob)
    print(f'  Connection OK — test image size: {img_test.size}')
except Exception as e:
    print(f'  Connection FAILED: {e}')
    raise

## 6. Final prompt (Option B)

Option B was selected through a prompt engineering evaluation comparing three
variants on 500 image pairs. It achieved the highest balanced accuracy (0.4357)
and Cohen kappa (0.1858) against filtered human votes.

In [ ]:
PROMPT_FINAL = (
    'You will see two images: Image A (left) and Image B (right). '
    'Which image looks safer for bicycle users? '
    'Choose 1 if Image A looks safer, 2 if Image B looks safer, '
    'or 0 if both look equally safe - many pairs look similarly safe '
    'and if there is no clear meaningful difference, choose 0.\n'
    'Respond with ONE line only:\n'
    '[VOTE]: [Vote Number] - [reason]\n\n'
    'Examples:\n'
    '[VOTE]: 1 - Image A has a physically separated cycle track and '
    'lower motor traffic volume, reducing conflict risk.\n'
    '[VOTE]: 2 - Image B has a wider road with lower motor traffic '
    'speed and no close vehicle passing.\n'
    '[VOTE]: 0 - Both streets are quiet residential roads with '
    'similar traffic volume and no dedicated cycling provision.\n\n'
)

print('Prompt loaded.')
print('=' * 60)
print(PROMPT_FINAL)
print('=' * 60)

## 7. GPT API call and vote parser

In [ ]:
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format='JPEG', quality=quality)
    return base64.b64encode(buff.getvalue()).decode('utf-8')


def call_gpt(prompt_text, pil1, pil2):
    uri1 = f'data:image/jpeg;base64,{pil_to_base64_jpeg(pil1)}'
    uri2 = f'data:image/jpeg;base64,{pil_to_base64_jpeg(pil2)}'
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text',      'text': 'Image A:'},
            {'type': 'image_url', 'image_url': {'url': uri1}},
            {'type': 'text',      'text': '\nImage B:'},
            {'type': 'image_url', 'image_url': {'url': uri2}},
            {'type': 'text',      'text': f'\n\n{prompt_text}'},
        ],
    }]
    response = client.chat.completions.create(
        model=OPENAI_MODEL_ID,
        messages=messages,
        max_completion_tokens=256,
    )
    return response.choices[0].message.content


def parse_vote(raw_txt):
    """
    Parse GPT response into (vote_number, reason, raw_text).
    Tries three patterns in order:
    1. [VOTE]: N - reason   (expected format)
    2. N - reason           (format without tag)
    3. N                    (bare number fallback)
    Returns (None, '', raw) if no pattern matches.
    """
    raw_clean = raw_txt.strip() if raw_txt else ''
    patterns  = [
        r'^\[VOTE\]\s*[:\-]\s*([0-2])\s*[:\-]\s*(.*)$',
        r'^([0-2])\s*[:\-]\s*(.*)$',
        r'([0-2])',
    ]
    for pat in patterns:
        m = re.search(pat, raw_clean, re.IGNORECASE | re.MULTILINE)
        if m:
            vote_num = int(m.group(1))
            reason   = m.group(2).strip() if len(m.groups()) >= 2 else ''
            return vote_num, reason, raw_clean
    return None, '', raw_clean


print('API client and parser ready.')

## 8. Run pairs

Output columns: `user_id`, `image_name1`, `image_name2`, `vote`, `location`,
`voting_time`, `model_id`, `reason`.

Image names include the full path prefix `training/training/Images/`.

Content policy violations are skipped immediately without retrying.
Checkpoints save to disk every `CHECKPOINT_EVERY` successful results.

Progress printed every 10 results: pairs done, percentage complete,
draw rate, skipped count and parse failure count.

In [ ]:
results        = []
parse_failures = []
skipped        = []

print(f'Running {total_pairs:,} new pairs with {OPENAI_MODEL_ID}...')
print(f'GPT user_id : {GPT_USER_ID}')
print('=' * 70)

for i, row in tqdm(sample_df.iterrows(), total=total_pairs, desc='Pairs'):
    name1 = row['image_name1']
    name2 = row['image_name2']

    # ── download images with retry ────────────────────────────────────────
    pil1 = pil2 = None
    for attempt in range(RETRY_MAX):
        try:
            pil1 = download_blob_to_pil(IMAGE_PREFIX + name1)
            pil2 = download_blob_to_pil(IMAGE_PREFIX + name2)
            break
        except Exception as e:
            if attempt == RETRY_MAX - 1:
                print(f'  SKIP (download) {name1} / {name2}: {e}')
                skipped.append({
                    'image_name1': IMAGE_PREFIX + name1,
                    'image_name2': IMAGE_PREFIX + name2,
                    'reason'     : f'download_failed: {str(e)[:80]}'
                })
            time.sleep(2 ** attempt)

    if pil1 is None or pil2 is None:
        continue

    # ── call GPT with retry ───────────────────────────────────────────────
    raw_txt   = None
    vote_time = datetime.now().strftime('%d/%m/%Y %H:%M')
    for attempt in range(RETRY_MAX):
        try:
            raw_txt   = call_gpt(PROMPT_FINAL, pil1, pil2)
            vote_time = datetime.now().strftime('%d/%m/%Y %H:%M')
            break
        except Exception as e:
            err_str = str(e)
            if 'content_policy_violation' in err_str or '400' in err_str:
                print(f'  SKIP (content policy) {name1} / {name2}')
                skipped.append({
                    'image_name1': IMAGE_PREFIX + name1,
                    'image_name2': IMAGE_PREFIX + name2,
                    'reason'     : 'content_policy_violation'
                })
                break
            if attempt == RETRY_MAX - 1:
                print(f'  SKIP (API) {name1} / {name2}: {err_str[:80]}')
                skipped.append({
                    'image_name1': IMAGE_PREFIX + name1,
                    'image_name2': IMAGE_PREFIX + name2,
                    'reason'     : f'api_failed: {err_str[:80]}'
                })
            time.sleep(2 ** attempt)

    if raw_txt is None:
        continue

    # ── parse response ────────────────────────────────────────────────────
    ai_vote, reason, raw_clean = parse_vote(raw_txt)

    if ai_vote is None:
        print(f'  PARSE FAIL {name1} / {name2}: {raw_clean[:80]}')
        parse_failures.append({
            'image_name1': IMAGE_PREFIX + name1,
            'image_name2': IMAGE_PREFIX + name2,
            'raw_txt'    : raw_clean
        })
        skipped.append({
            'image_name1': IMAGE_PREFIX + name1,
            'image_name2': IMAGE_PREFIX + name2,
            'reason'     : f'parse_failed: {raw_clean[:80]}'
        })
        continue

    # ── clean encoding ────────────────────────────────────────────────────
    reason = reason.encode('utf-8', errors='replace').decode('utf-8')

    # ── save result ───────────────────────────────────────────────────────
    results.append({
        'user_id'     : GPT_USER_ID,
        'image_name1' : IMAGE_PREFIX + name1,
        'image_name2' : IMAGE_PREFIX + name2,
        'vote'        : ai_vote,
        'location'    : '',
        'voting_time' : vote_time,
        'model_id'    : OPENAI_MODEL_ID,
        'reason'      : reason,
    })

    # ── progress every 10 results ─────────────────────────────────────────
    if len(results) % 10 == 0:
        pct = len(results) / total_pairs * 100
        dr  = sum(1 for r in results if r['vote'] == 0) / len(results) * 100
        print(
            f'  [{len(results):>6} / {total_pairs:,}]  '
            f'{pct:>5.1f}%  '
            f'draw={dr:.1f}%  '
            f'skipped={len(skipped)}  '
            f'failures={len(parse_failures)}'
        )

    # ── checkpoint ───────────────────────────────────────────────────────
    if len(results) % CHECKPOINT_EVERY == 0 and len(results) > 0:
        pd.DataFrame(results).to_csv(
            RESULT_CSV_PATH, index=False, encoding='utf-8-sig'
        )
        print(f'  Checkpoint saved: {len(results):,} results')

    time.sleep(RATE_LIMIT_SEC)

print('\n' + '=' * 70)
print('Run complete.')
print(f'  Processed      : {len(results):,}')
print(f'  Parse failures : {len(parse_failures)}')
print(f'  Skipped        : {len(skipped)}')

## 9. Export and summary

In [ ]:
if not results:
    print('No results to save. Check the error messages above.')
else:
    final_df  = pd.DataFrame(results)
    final_df.to_csv(RESULT_CSV_PATH, index=False, encoding='utf-8-sig')

    n         = len(final_df)
    draw_rate = (final_df['vote'] == 0).mean()
    vote_dist = final_df['vote'].value_counts(normalize=True).sort_index()

    print('=' * 60)
    print('SUMMARY')
    print('=' * 60)
    print(f'  Saved to       : {RESULT_CSV_PATH}')
    print(f'  New pairs      : {n:,}')
    print(f'  Parse failures : {len(parse_failures)}')
    print(f'  Skipped        : {len(skipped)}')
    print(f'  GPT draw rate  : {draw_rate*100:.2f}%')
    print()
    print('  Vote distribution:')
    for v, pct in vote_dist.items():
        bar = '#' * int(pct * 30)
        print(f'    {int(v)} : {bar:<30} {pct*100:.1f}%')
    print()
    print('  Output columns:')
    print(f'    {list(final_df.columns)}')
    print()
    print('  Sample output (first 3 rows):')
    print(final_df[['user_id','image_name1','vote',
                    'voting_time','model_id']]
          .head(3).to_string(index=False))
    print()

    if skipped:
        print(f'\nSkipped pairs ({len(skipped)}):')
        from collections import Counter
        reasons = Counter(s['reason'].split(':')[0] for s in skipped)
        for reason, count in reasons.most_common():
            print(f'  {reason}: {count}')